# DEH 30-Day PySpark Challenge
## Day 3 — Schema & DataFrame Reader

**Instructions:**
1. Click `File → Save a copy in Drive` before editing
2. Run each cell in order using `Shift + Enter`
3. Complete the assignment cells at the bottom

---

In [1]:
!pip install pyspark --quiet

In [2]:
import os

os.environ['AWS_ACCESS_KEY_ID']     = ''
os.environ['AWS_SECRET_ACCESS_KEY'] = ''
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'

BUCKET = 'deh-pyspark-challenge-<account id>'

print('Credentials set.')


Credentials set.


In [3]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, DateType, LongType, BooleanType, TimestampType
)

spark = (
    SparkSession.builder
    .appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

Spark version: 4.0.3


## Part 1 — Schema

### Step 4 — Default read (no schema)

In [ ]:
# Default read — everything is StringType
orders_default = spark.read.csv(f"s3a://{BUCKET}/data/orders.csv", header=True)
print("--- Default schema ---")
orders_default.printSchema()

--- Default schema ---
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_pct: string (nullable = true)
 |-- status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- region: string (nullable = true)



### Step 5 — Explicit schema with StructType

In [7]:
orders_schema = StructType([
    StructField("order_id",       StringType(),   True),
    StructField("customer_id",    StringType(),   True),
    StructField("product_id",     StringType(),   True),
    StructField("order_date",     DateType(),     True),
    StructField("quantity",       IntegerType(),  True),
    StructField("unit_price",     DoubleType(),   True),
    StructField("discount_pct",   IntegerType(),  True),
    StructField("status",         StringType(),   True),
    StructField("payment_method", StringType(),   True),
    StructField("region",         StringType(),   True)
])

orders_df = spark.read.csv(
    f"s3a://{BUCKET}/data/orders.csv",
    header=True,
    schema=orders_schema
)

print("--- Explicit schema ---")
orders_df.printSchema()
orders_df.show(3)

--- Explicit schema ---
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_pct: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- region: string (nullable = true)

+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+
|order_id|customer_id|product_id|order_date|quantity|unit_price|discount_pct|   status|payment_method| region|
+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+
|   O0001|       C001|      P001|2023-01-05|       2|   1299.99|          10|Delivered|   Credit Card|   East|
|   O0002|       C002|      P005|2023-01-07|       1|    449.99|           0|Delivered|        PayPal|   West|


## Part 2 — DataFrame Reader API

### Step 6 — Two styles of passing options

In [ ]:
# Style 1 — keyword arguments
df1 = spark.read.csv(f"s3a://{BUCKET}/data/orders.csv", header=True, inferSchema=True)

# Style 2 — chained .option() calls (preferred in production)
df2 = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"s3a://{BUCKET}/data/orders.csv")

print(f"Both styles same result: {df1.count() == df2.count()}")

Both styles same result: True


### Step 7 — Reading with multiple options and explicit schema

In [ ]:
orders_df = spark.read \
    .option("header", "true") \
    .option("dateFormat", "yyyy-MM-dd") \
    .option("nullValue", "N/A") \
    .option("mode", "PERMISSIVE") \
    .schema(orders_schema) \
    .csv(f"s3a://{BUCKET}/data/orders.csv")

orders_df.printSchema()
orders_df.show(3)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_pct: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- region: string (nullable = true)

+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+
|order_id|customer_id|product_id|order_date|quantity|unit_price|discount_pct|   status|payment_method| region|
+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+
|   O0001|       C001|      P001|2023-01-05|       2|   1299.99|          10|Delivered|   Credit Card|   East|
|   O0002|       C002|      P005|2023-01-07|       1|    449.99|           0|Delivered|        PayPal|   West|
|   O0003|       C003|  

## Part 3 — Reading Modes

### Step 8 — PERMISSIVE with _corrupt_record (on dirty data)

In [4]:
schema_with_corrupt = StructType([
    StructField("order_id",        StringType(),  True),
    StructField("customer_id",     StringType(),  True),
    StructField("product_id",      StringType(),  True),
    StructField("order_date",      DateType(),    True),
    StructField("quantity",        IntegerType(), True),
    StructField("unit_price",      DoubleType(),  True),
    StructField("discount_pct",    IntegerType(), True),
    StructField("status",          StringType(),  True),
    StructField("payment_method",  StringType(),  True),
    StructField("region",          StringType(),  True),
    StructField("_corrupt_record", StringType(),  True)
])

df_permissive = spark.read \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .schema(schema_with_corrupt) \
    .csv(f"s3a://{BUCKET}/data/orders_dirty.csv")



df_permissive.show()


+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+--------------------+
|order_id|customer_id|product_id|order_date|quantity|unit_price|discount_pct|   status|payment_method| region|     _corrupt_record|
+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+--------------------+
|   O0001|       C001|      P001|2023-01-05|       2|   1299.99|          10|Delivered|   Credit Card|   East|                NULL|
|   O0002|       C002|      P005|2023-01-07|       1|    449.99|           0|Delivered|        PayPal|   West|                NULL|
|   O0003|       C003|      P003|2023-01-10|       4|    349.99|          15|Delivered|   Credit Card|Midwest|                NULL|
|   O0004|       C004|      P004|2023-01-12|    NULL|    199.99|           5|  Shipped|   Credit Card|  South|O0004,C004,P004,2...|
|   O0005|       C005|      P002|2023-01-15|       3|     29.99|           0

In [5]:
df_permissive.show(truncate=False)

+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+-------------------------------------------------------------------+
|order_id|customer_id|product_id|order_date|quantity|unit_price|discount_pct|status   |payment_method|region |_corrupt_record                                                    |
+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+-------------------------------------------------------------------+
|O0001   |C001       |P001      |2023-01-05|2       |1299.99   |10          |Delivered|Credit Card   |East   |NULL                                                               |
|O0002   |C002       |P005      |2023-01-07|1       |449.99    |0           |Delivered|PayPal        |West   |NULL                                                               |
|O0003   |C003       |P003      |2023-01-10|4       |349.99    |15          |Delivered|Credit Card   |Mid

### Step 9 — DROPMALFORMED

In [8]:
df_drop = spark.read \
    .option("header", "true") \
    .option("mode", "DROPMALFORMED") \
    .schema(orders_schema) \
    .csv(f"s3a://{BUCKET}/data/orders_dirty.csv")


df_drop.show()

+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+
|order_id|customer_id|product_id|order_date|quantity|unit_price|discount_pct|   status|payment_method| region|
+--------+-----------+----------+----------+--------+----------+------------+---------+--------------+-------+
|   O0001|       C001|      P001|2023-01-05|       2|   1299.99|          10|Delivered|   Credit Card|   East|
|   O0002|       C002|      P005|2023-01-07|       1|    449.99|           0|Delivered|        PayPal|   West|
|   O0003|       C003|      P003|2023-01-10|       4|    349.99|          15|Delivered|   Credit Card|Midwest|
|   O0005|       C005|      P002|2023-01-15|       3|     29.99|           0|Delivered|   Credit Card|   West|
|   O0006|       C006|      P008|2023-01-18|       1|    199.99|          10|Delivered|   Credit Card|   East|
|   O0008|       C008|      P014|2023-01-22|       1|     59.99|           0|Delivered|   Credit Card|   West|
|

### Step 10 — FAILFAST

In [9]:
df_strict = spark.read \
    .option("header", "true") \
    .option("mode", "FAILFAST") \
    .schema(orders_schema) \
    .csv(f"s3a://{BUCKET}/data/orders_dirty.csv")

df_strict.show()
# Throws SparkException immediately on the first bad row

Py4JJavaError: An error occurred while calling o76.showString.
: org.apache.spark.SparkException: [FAILED_READ_FILE.NO_HINT] Encountered error while reading file s3a://deh-pyspark-challenge-030798167757/data/orders_dirty.csv.  SQLSTATE: KD001
	at org.apache.spark.sql.errors.QueryExecutionErrors$.cannotReadFilesError(QueryExecutionErrors.scala:856)
	at org.apache.spark.sql.execution.datasources.v2.FileDataSourceV2$.attachFilePath(FileDataSourceV2.scala:142)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:142)
	at scala.collection.Iterator$$anon$9.hasNext(Iterator.scala:583)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext(WholeStageCodegenEvaluatorFactory.scala

---
## Assignment — Day 3

---

### Task 1
Read `products.csv` from S3 with no options. Print the schema.  
Then read it again with `inferSchema=True`. What columns changed type?

In [ ]:
# Task 1 — Write your code here


### Task 2
Define an explicit `StructType` schema for `products.csv`:  
`product_id` (String), `product_name` (String), `category` (String), `sub_category` (String), `unit_price` (Double), `cost_price` (Double), `supplier` (String), `stock_quantity` (Integer).  
Read the file and verify the schema.

In [ ]:
# Task 2 — Write your code here


### Task 3
Read `customers.csv` from S3 using the chained `.option()` style with an explicit schema.  
Columns: `customer_id`, `first_name`, `last_name`, `email`, `city`, `state`, `country`, `signup_date` (Date), `segment`.  
Show the first 5 rows.

In [ ]:
# Task 3 — Write your code here


### Task 4
Read `orders_dirty.csv` using PERMISSIVE mode with `_corrupt_record`.  
Print total rows and bad row count.  
Then read it with DROPMALFORMED — compare the counts.  
Why does DROPMALFORMED still return 15 rows for type mismatches but drops the 3 column-count rows?

In [ ]:
# Task 4 — Write your code here


---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*